## Import Libraries and Some Function

In [1]:
import pandas
import numpy
import random
import requests
import time
import re
from urllib.parse import urljoin

from bs4 import BeautifulSoup

In [2]:
BASE_URL = "https://www.vlr.gg"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/142.0")
}

In [3]:
def absolute(base_url: str = BASE_URL, url: str = None):
    return urljoin(base=base_url, url=url)

In [4]:
def get_soup(
        url: str,
        headers: dict = None,
        retry: int = 3,
        delay_range: set = (5, 10)) -> None:
    session = requests.session()
    session.headers.update(headers=headers)
    for attempt in range(retry):
        try:
                response = session.get(url=url, timeout=10)
                response.raise_for_status()
                if response.status_code == 429:
                    time.sleep(300)
                    continue

                time.sleep(random.uniform(25, 30))
                return BeautifulSoup(response.content, "html.parser")
        except Exception as e:
            print(f"Attempt {attempt + 1} Failed: {e}")
            time.sleep(random.uniform(*delay_range))
    
    print(f"failed to get soup for {url} after {retry} attempts.")
    return None
    

## Getting list of tournament

In [5]:
soup = get_soup(url=BASE_URL)
contents = soup.select(".header-inner a[href]")
for content in contents:
    href = content.get("href")
    if "events" not in href:
        continue
    url = absolute(url=href)

soup = get_soup(url=url)
contents = soup.select(".wf-filter-inner a[href]")
for content in contents:
    href = content.get("href")
    if "60" not in href:
        continue
    url = absolute(url=href)

soup = get_soup(url=url)
contents = soup.select(".events-container-col a[href]")
results = set()
for content in contents:
    href = content.get("href")
    url = absolute(url=href)
    results.add(url)

results = list(results)
results[:5]

['https://www.vlr.gg/event/2094/champions-tour-2024-emea-stage-2',
 'https://www.vlr.gg/event/2684/vct-2026-emea-kickoff',
 'https://www.vlr.gg/event/1926/champions-tour-2024-china-kickoff',
 'https://www.vlr.gg/event/2380/vct-2025-emea-stage-1',
 'https://www.vlr.gg/event/2450/china-evolution-series-act-2-x-asian-champions-league']

## Getting Match Description

In [6]:
soup = get_soup("https://www.vlr.gg/event/2683/vct-2026-pacific-kickoff")
title_container = soup.select_one(".wf-title")
title = title_container.get_text().strip()
desc = soup.select(".event-desc-inner a[href]")
tour = desc[0].get_text().strip()
if "valorant champions tour" in tour.lower():
    stage = desc[1].get_text().strip()
    region = desc[2].get_text().strip()
    print(f"This is a {stage} stage on {tour} in {region} region.")
    contents = soup.select(".wf-nav a[href]")
    matches_page = set()
    stats_page = set()
    agents_page = set()
    for content in contents:
        href = content.get("href")
        url = absolute(url=href)
        if "matches" in href:
            matches_page.add(url)
        elif "stats" in href:
            stats_page.add(url)
        elif "agents" in href:
            agents_page.add(url)
        else:
            continue

    for variable in [matches_page, stats_page, agents_page]:
        variable = list(variable)
        print(f"Url exists in {variable}" if len(variable) > 0 else f"Url not exists in {variable}")

This is a Kickoff stage on Valorant Champions Tour 2026 in Pacific region.
Url exists in ['https://www.vlr.gg/event/matches/2683/vct-2026-pacific-kickoff/?series_id=5222']
Url exists in ['https://www.vlr.gg/event/stats/2683/vct-2026-pacific-kickoff']
Url exists in ['https://www.vlr.gg/event/agents/2683/vct-2026-pacific-kickoff']


In [7]:
soup = get_soup("https://www.vlr.gg/event/matches/2683/vct-2026-pacific-kickoff/?series_id=5222")
match_containers = soup.select(".wf-card a[href]")
matches = set()
for match in match_containers:
    href = match.get("href")
    if "5956" not in href:
        continue
    url = absolute(url=href)
    matches.add(url)

result = list(matches)
result

['https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1',
 'https://www.vlr.gg/595656/paper-rex-vs-global-esports-vct-2026-pacific-kickoff-lr3',
 'https://www.vlr.gg/595654/global-esports-vs-gen-g-vct-2026-pacific-kickoff-lr2',
 'https://www.vlr.gg/595637/rex-regum-qeon-vs-detonation-focusme-vct-2026-pacific-kickoff-ur2',
 'https://www.vlr.gg/595636/paper-rex-vs-global-esports-vct-2026-pacific-kickoff-ur2',
 'https://www.vlr.gg/595644/gen-g-vs-t1-vct-2026-pacific-kickoff-mr1',
 'https://www.vlr.gg/595645/detonation-focusme-vs-global-esports-vct-2026-pacific-kickoff-mr2',
 'https://www.vlr.gg/595651/team-secret-vs-zeta-division-vct-2026-pacific-kickoff-lr1',
 'https://www.vlr.gg/595648/paper-rex-vs-t1-vct-2026-pacific-kickoff-mr3',
 'https://www.vlr.gg/595641/team-secret-vs-detonation-focusme-vct-2026-pacific-kickoff-mr1',
 'https://www.vlr.gg/595657/drx-vs-paper-rex-vct-2026-pacific-kickoff-lr4',
 'https://www.vlr.gg/595643/varrel-vs-drx-vct-2026-pacif

### Getting Match Details

In [8]:
soup = get_soup("https://www.vlr.gg/595630/nongshim-redforce-vs-team-secret-vct-2026-pacific-kickoff-ur1")
bracket_info_container = soup.select_one(".match-header-event-series")
bracket_name = bracket_info_container.get_text().split(":")[1].strip()
date_info_container = soup.select_one(".moment-tz-convert")
date = date_info_container.get("data-utc-ts")
patch_info_container = soup.select_one(".match-header-date")
patch = re.sub(r"[\n\t]", "", patch_info_container.get_text()).split("Patch")[-1].strip()
print(f"Match date: {date}")
print(f"Match bracket: {bracket_name}")
print(f"Patch info: {patch}")

Match date: 2026-01-22 03:00:00
Match bracket: Upper Round 1
Patch info: 12.0


In [17]:
home_team_container = soup.select_one(".match-header-link.wf-link-hover.mod-1")
home_href = home_team_container.get("href")
home_url = absolute(url=home_href)
home_soup = get_soup(home_url)
home_container = home_soup.select('.wf-title')
home_name = home_container[0].get_text()
home_alias = home_container[1].get_text()

away_team_container = soup.select_one(".match-header-link.wf-link-hover.mod-2")
away_href = away_team_container.get("href")
away_url = absolute(url=away_href)
away_soup = get_soup(away_url)
away_container = away_soup.select('.wf-title')
away_name = away_container[0].get_text()
away_alias = away_container[1].get_text()

bo_info_container = soup.select(".match-header-vs-note")
bo_info = bo_info_container[1].get_text().strip()
score_container = soup.select_one(".js-spoiler")
home_score = re.sub(r"[\n\t]", "", score_container.get_text()).split(":")[0].strip()
away_score = re.sub(r"[\n\t]", "", score_container.get_text()).split(":")[-1].strip()

print(f"match -> {home_name} ({home_alias}) vs ({away_alias}) {away_name}")
print(f"score -> {home_score} : {away_score}")
print(f"BO -> {bo_info}")

match -> Nongshim RedForce (NS) vs (TS) Team Secret
score -> 2 : 0
BO -> Bo3


In [25]:
map_selection_container = soup.select_one(".match-header-note")
map_selection = map_selection_container.get_text().split(";")
home_map_picks, home_map_bans = [], []
away_map_picks, away_map_bans = [], []
decider_map = []
for map in map_selection:
    map_split = map.strip().split(" ")
    # print(map_split)
    if map_split[0] == home_alias:
        if map_split[1] == "pick":
            home_map_picks.append(map_split[2].strip())
        else:
            home_map_bans.append(map_split[2].strip())
    elif map_split[0] == away_alias:
        if map_split[1] == "pick":
            away_map_picks.append(map_split[2].strip())
        else:
            away_map_bans.append(map_split[2].strip())
    else:
        decider_map.append(map_split[0].strip())

In [39]:
head_to_head_container = soup.select(".match-h2h-matches-score")
home_h2h_win = 0
away_h2h_win = 0
home_h2h_score = 0
away_h2h_score = 0
for container in head_to_head_container:
    score = container.get_text().strip().split("\n")
    home_h2h_score += int(score[0])
    away_h2h_score += int(score[1])
    if int(score[0]) > int(score[1]):
        home_h2h_win += 1
    else:
        away_h2h_win += 1

print(f"Home head-to-head win: {home_h2h_win}")
print(f"Away head-to-head win: {away_h2h_win}")
print(f"Home head-to-head score: {home_h2h_score}")
print(f"Away head-to-head score: {away_h2h_score}")

Home head-to-head win: 2
Away head-to-head win: 1
Home head-to-head score: 4
Away head-to-head score: 2


In [52]:
last_match_container = soup.select(".wf-card.mod-dark.match-histories")
home_last_match = last_match_container[0].select(".match-histories-item.wf-module-item")
home_last_match_win = 0
for match in home_last_match:
    match_class = match.get("class")
    if "mod-win" not in match_class:
        continue
    home_last_match_win += 1
    home_last_match_wr = home_last_match_win / 5

away_last_match = last_match_container[1].select(".match-histories-item.wf-module-item")
away_last_match_win = 0
for match in away_last_match:
    match_class = match.get("class")
    if "mod-win" not in match_class:
        continue
    away_last_match_win += 1
    away_last_match_wr = away_last_match_win / 5

print(f"Home winrate last 5 matches: {home_last_match_wr}")
print(f"Away winrate last 5 matches: {away_last_match_wr}")

Home winrate last 5 matches: 0.8
Away winrate last 5 matches: 0.4


### Getting Game Details